In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install unsloth

In [5]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

messages = [{"role": "user", "content": "What is 12 * 8?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0]))

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="train")

def format_prompt(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_prompt)
print(dataset[0]["text"])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=SFTConfig(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("phi35_mini_gsm8k_lora")
tokenizer.save_pretrained("phi35_mini_gsm8k_lora")

In [ ]:
 !pip install -U torchao -q 

In [ ]:
import json
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

NUM_QUESTIONS = 100
CANDIDATES_PER_CHECKPOINT = 4
OUTPUT_PATH = "verifier_training_data_v2.jsonl"

GEN_BASE = "microsoft/Phi-3.5-mini-instruct"
CHECKPOINT_PATHS = {
    "checkpoint-935": "/kaggle/input/datasets/naitiksudo/phi35-gsm8k-checkpoints/outputs/checkpoint-935",
    "checkpoint-500": "/kaggle/input/datasets/naitiksudo/phi35-gsm8k-checkpoints/outputs/checkpoint-500",
}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def extract_answer(text):
    match = re.search(r"####\s*(-?[\d,]+\.?\d*)", text)
    return match.group(1).replace(",", "").strip() if match else None

def load_checkpoint(adapter_path):
    tok = AutoTokenizer.from_pretrained(GEN_BASE)
    base = AutoModelForCausalLM.from_pretrained(
        GEN_BASE, quantization_config=bnb_config, device_map="auto",
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()
    return model, tok

def generate(model, tokenizer, question, n=4, temperature=0.7, top_p=0.9):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=256, do_sample=True,
        temperature=temperature, top_p=top_p,
        num_return_sequences=n, pad_token_id=tokenizer.eos_token_id,
    )
    return [tokenizer.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for o in outputs]

dataset = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=42).select(range(NUM_QUESTIONS))

records = []
for ckpt_name, ckpt_path in CHECKPOINT_PATHS.items():
    print(f"Loading {ckpt_name}...")
    model, tokenizer = load_checkpoint(ckpt_path)

    for i, ex in enumerate(dataset):
        question, gold_solution = ex["question"], ex["answer"]
        gold = extract_answer(gold_solution)
        candidates = generate(model, tokenizer, question, n=CANDIDATES_PER_CHECKPOINT)
        for c in candidates:
            pred = extract_answer(c)
            label = int(pred is not None and pred == gold)
            records.append({"question": question, "candidate": c, "label": label, "source_checkpoint": ckpt_name})
        if (i + 1) % 50 == 0:
            print(f"  {ckpt_name}: {i+1}/{NUM_QUESTIONS} done")

    del model
    torch.cuda.empty_cache()

with open(OUTPUT_PATH, "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(records)} records to {OUTPUT_PATH}")

In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets scikit-learn

import json
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 67.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.


In [3]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # float16, not bf16 — this session's GPU doesn't support bf16
    bnb_4bit_use_double_quant=True,
)

In [ ]:
!find /kaggle/input -iname "verifier_training_data.jsonl"

In [ ]:
DATA_PATH = "/kaggle/input/datasets/naitiksudo/verifier-training-data/verifier_training_data.jsonl"  # replace with the path from Cell 2

records = []
with open(DATA_PATH, "r") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} records")
print(records[0])

In [ ]:
from datasets import ClassLabel

def normalize_label(label):
    if isinstance(label, str):
        return 1 if label.strip().lower() in ("correct", "1", "true") else 0
    return int(label)

def build_text(r):
    return f"Question: {r['question']}\n\nCandidate Solution: {r['candidate']}"

for r in records:
    r["text"] = build_text(r)
    r["label"] = normalize_label(r["label"])

ds = Dataset.from_list([{"text": r["text"], "label": r["label"]} for r in records])

# cast label to ClassLabel so stratify_by_column works
ds = ds.cast_column("label", ClassLabel(num_classes=2, names=["incorrect", "correct"]))

ds = ds.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, eval_ds = ds["train"], ds["test"]
print(train_ds)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
MAX_LEN = 512

def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
    out["labels"] = batch["label"]
    return out

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_fn, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="/kaggle/working/verifier_outputs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
model.save_pretrained("/kaggle/working/llama32_1b_verifier_lora")
tokenizer.save_pretrained("/kaggle/working/llama32_1b_verifier_lora")
print("Saved to /kaggle/working/llama32_1b_verifier_lora")

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [4]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

GEN_BASE = "microsoft/Phi-3.5-mini-instruct"
GEN_ADAPTER = "/kaggle/input/datasets/naitiksudo/dpo-dataset/phi35_mini_gsm8k_dpo_lora"  

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_BASE)
gen_base = AutoModelForCausalLM.from_pretrained(
    GEN_BASE,
    quantization_config=bnb_config,  # reuse the BitsAndBytesConfig from Cell 5
    device_map="auto",
)
gen_model = PeftModel.from_pretrained(gen_base, GEN_ADAPTER)
gen_model.eval()

config.json: 0.00B [00:00, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_proj): Linear4

In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

VERIFIER_BASE = "meta-llama/Llama-3.2-1B-Instruct"
VERIFIER_ADAPTER = "Naitik-sudo123/llama32-1b-gsm8k-verifier-lora"

tokenizer = AutoTokenizer.from_pretrained(VERIFIER_ADAPTER)  # saved alongside the adapter
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

verifier_base = AutoModelForSequenceClassification.from_pretrained(
    VERIFIER_BASE,
    num_labels=2,
    quantization_config=bnb_config,   # reuse from earlier cell
    device_map="auto",
)
verifier_base.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(verifier_base, VERIFIER_ADAPTER)
model.eval()

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): LlamaForSequenceClassification(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )

In [7]:
from datasets import load_dataset

gsm8k_test = load_dataset("gsm8k", "main", split="test")
eval_subset = gsm8k_test.shuffle(seed=42).select(range(50))

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [19]:
import re

_FINAL_RE = re.compile(r"####\s*(-?[\d,\.]+)")
_LAST_NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")  # only counts '.' as decimal if digits follow it

def extract_answer(text):
    m = _FINAL_RE.search(text)
    if m:
        return m.group(1).replace(",", "").strip()

    stripped = text.rstrip()
    if not stripped or stripped[-1] not in ".!)":
        return None  # looks cut off mid-sentence — don't guess

    nums = _LAST_NUM_RE.findall(stripped)
    return nums[-1].replace(",", "") if nums else None

def generate(question, n=1, do_sample=True, temperature=0.7):
    prompt = f"Question: {question}\nAnswer:"
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=0.9 if do_sample else None,
        num_return_sequences=n,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return [gen_tokenizer.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for o in outputs]

In [20]:
MAX_LEN = 512

def build_text(r):
    return f"Question: {r['question']}\n\nCandidate Solution: {r['candidate']}"

def score_candidate(question, candidate):
    text = build_text({"question": question, "candidate": candidate})
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1)[0, 1].item()  # P(correct)

In [21]:
import re
import ast
import operator

# --- Safe arithmetic evaluator (no eval(), only +-*/ and parens) ---
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Disallowed expression: {ast.dump(node)}")

def safe_arithmetic_eval(expr: str) -> float:
    expr = expr.replace(",", "").strip()
    return _safe_eval(ast.parse(expr, mode="eval").body)


_CALC_RE = re.compile(r"<<([^=<>]+)=([^<>]+)>>")
_FINAL_RE = re.compile(r"####\s*(-?[\d,\.]+)")

def verify_arithmetic(candidate_text: str, tol: float = 1e-3) -> dict:
    """Recomputes every <<expr=result>> annotation and cross-checks the
    final '#### answer' against the last verified step."""
    checks = _CALC_RE.findall(candidate_text)
    errors = []
    last_recomputed = None

    for expr, stated in checks:
        try:
            recomputed = safe_arithmetic_eval(expr)
            stated_val = float(stated.replace(",", "").strip())
            last_recomputed = recomputed
            if abs(recomputed - stated_val) > tol:
                errors.append((expr.strip(), stated_val, recomputed))
        except Exception:
            errors.append((expr.strip(), stated.strip(), None))

    final_match = _FINAL_RE.search(candidate_text)
    final_answer_ok = None
    if final_match and last_recomputed is not None:
        try:
            stated_final = float(final_match.group(1).replace(",", "").strip())
            final_answer_ok = abs(stated_final - last_recomputed) <= tol
        except ValueError:
            final_answer_ok = False

    return {
        "calc_valid": len(errors) == 0,
        "num_checks": len(checks),
        "num_errors": len(errors),
        "errors": errors,
        "final_answer_ok": final_answer_ok,
    }


def select_best_candidate(question, candidates, verifier_score_fn):
    """Calculator check runs first as a cheap veto; verifier only scores
    the arithmetically-valid survivors. Falls back to the full candidate
    set if none survive, so a real attempt is never just thrown away."""
    checked = [(c, verify_arithmetic(c)) for c in candidates]
    valid_pool = [c for c, r in checked if r["calc_valid"]]
    pool = valid_pool if valid_pool else candidates

    scored = [(c, verifier_score_fn(question, c)) for c in pool]
    best_candidate, best_score = max(scored, key=lambda x: x[1])

    return {
        "best_candidate": best_candidate,
        "best_score": best_score,
        "num_calc_valid": len(valid_pool),
        "num_total": len(candidates),
    }

In [ ]:
N = 4
correct_bon, correct_greedy, total = 0, 0, 0

for ex in eval_subset:
    question, gold = ex["question"], extract_answer(ex["answer"])

    candidates = generate(question, n=N, do_sample=True)
    result = select_best_candidate(question, candidates, score_candidate)
    best_candidate = result["best_candidate"]

    greedy = generate(question, n=1, do_sample=False)[0]

    correct_bon += int(extract_answer(best_candidate) == gold)
    correct_greedy += int(extract_answer(greedy) == gold)
    total += 1

print(f"Best-of-{N} + verifier: {correct_bon/total:.2%}")
print(f"Greedy baseline:        {correct_greedy/total:.2%}")

In [ ]:
from unsloth import FastLanguageModel
import torch, json
from datasets import load_dataset, Dataset, concatenate_datasets
from trl import SFTTrainer, SFTConfig 

DATA_PATH = "/kaggle/input/datasets/naitiksudo/verifier-training-data/verifier_training_data.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct", max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

orig = load_dataset("openai/gsm8k", "main", split="train")
def format_orig(example):
    messages = [{"role": "user", "content": example["question"]}, {"role": "assistant", "content": example["answer"]}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}
orig = orig.map(format_orig)

self_gen = [json.loads(l) for l in open(DATA_PATH) if json.loads(l)["label"] == 1]
print(f"RFT self-generated additions: {len(self_gen)} correct candidates")

def format_self_gen(r):
    messages = [{"role": "user", "content": r["question"]}, {"role": "assistant", "content": r["candidate"]}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}
self_gen_ds = Dataset.from_list(self_gen).map(format_self_gen)

rft_dataset = concatenate_datasets([orig.remove_columns([c for c in orig.column_names if c != "text"]), self_gen_ds])

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=rft_dataset,
    dataset_text_field="text", max_seq_length=2048,
    args=SFTConfig(
        per_device_train_batch_size=8, gradient_accumulation_steps=1, warmup_steps=10,
        num_train_epochs=1, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="rft_outputs",
    ),
)
trainer.train()
model.save_pretrained("phi35_mini_gsm8k_rft_lora")
tokenizer.save_pretrained("phi35_mini_gsm8k_rft_lora")

In [ ]:
pip install -U --no-cache-dir unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
import json, random, torch
from collections import defaultdict
from datasets import Dataset

DATA_PATH = "/kaggle/input/datasets/naitiksudo/verifier-training-data/verifier_training_data.jsonl"
RFT_ADAPTER = "phi35_mini_gsm8k_rft_lora"

records = [json.loads(l) for l in open(DATA_PATH)]
by_q = defaultdict(lambda: {"correct": [], "incorrect": []})
for r in records:
    by_q[r["question"]]["correct" if r["label"] == 1 else "incorrect"].append(r["candidate"])

pairs = [{"prompt": q, "chosen": random.choice(b["correct"]), "rejected": random.choice(b["incorrect"])}
         for q, b in by_q.items() if b["correct"] and b["incorrect"]]
print(f"DPO pairs: {len(pairs)} from {len(by_q)} unique questions")

model, tokenizer = FastLanguageModel.from_pretrained(model_name=RFT_ADAPTER, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_training(model)

def format_dpo(ex):
    ex["prompt"] = tokenizer.apply_chat_template([{"role": "user", "content": ex["prompt"]}], tokenize=False, add_generation_prompt=True)
    return ex
dpo_dataset = Dataset.from_list(pairs).map(format_dpo)

dpo_trainer = DPOTrainer(
    model=model, ref_model=None, tokenizer=tokenizer, train_dataset=dpo_dataset,
    args=DPOConfig(
        per_device_train_batch_size=4, gradient_accumulation_steps=2, warmup_steps=5,
        num_train_epochs=1, learning_rate=5e-6,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5, optim="adamw_8bit", seed=3407, output_dir="dpo_outputs", beta=0.1,
    ),
)
dpo_trainer.train()
model.save_pretrained("phi35_mini_gsm8k_dpo_lora")
tokenizer.save_pretrained("phi35_mini_gsm8k_dpo_lora")

In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

GEN_BASE = "microsoft/Phi-3.5-mini-instruct"
GEN_ADAPTER = "Naitik-sudo123/phi35-mini-gsm8k-dpo-lora"   # swap in — was phi35_mini_gsm8k_lora before

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_BASE)
gen_base = AutoModelForCausalLM.from_pretrained(GEN_BASE, quantization_config=bnb_config, device_map="auto")
gen_model = PeftModel.from_pretrained(gen_base, GEN_ADAPTER)
gen_model.eval()

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_proj): Linear4

In [ ]:
"""
push_to_hf.py

Uploads the rehearsal project's LoRA adapters and the verifier
training dataset to the Hugging Face Hub. Written to run from a
Kaggle notebook cell (or as a plain script) once your datasets are
attached / your working directory has the trained adapters in it.

Covers, out of the box:
  - phi35_mini_gsm8k_lora        (base QLoRA generator adapter)
  - phi35_mini_gsm8k_rft_lora    (RFT-tuned generator adapter)
  - phi35_mini_gsm8k_dpo_lora    (DPO-tuned generator adapter, final)
  - llama32_1b_verifier_lora     (verifier adapter)
  - verifier_training_data.jsonl (verifier training dataset)

BEFORE RUNNING:
  1. pip install -U huggingface_hub
  2. Give it a token: either
       a) add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets), or
       b) let it fall through to an interactive notebook_login()
  3. Fix the paths in CONFIG below — they're my best guess from what
     you've described, but I can't see your actual Kaggle file tree.
     Run `!find /kaggle/input -maxdepth 3` and `!find /kaggle/working
     -maxdepth 2` in a cell first if you're not sure, especially for
     the RFT/DPO adapters — you mentioned those were retrained after
     a session reset, so double-check whether they landed back in a
     committed dataset or are still sitting in /kaggle/working.
"""

import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo, notebook_login

# ---------------------------------------------------------------------------
# 1. Auth
# ---------------------------------------------------------------------------
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("writing token")
except Exception:
    HF_TOKEN = os.environ.get("writing token")

if not HF_TOKEN:
    print("No HF_TOKEN secret found — falling back to interactive login.")
    notebook_login()  # opens a token-paste prompt in the notebook output

api = HfApi(token=HF_TOKEN)

# ---------------------------------------------------------------------------
# 2. Config — EDIT THESE to match your real paths and desired repo names
# ---------------------------------------------------------------------------
HF_USERNAME = "Naitik-sudo123"   # change if your HF username differs from Kaggle
PRIVATE = False              # flip to True if you don't want these public yet

MODEL_ITEMS = [
    {
        "local_path": "/kaggle/input/datasets/naitiksudo/phi35-gsm8k-checkpoints/phi35_mini_gsm8k_lora",
        "repo_id": f"{HF_USERNAME}/phi35-mini-gsm8k-lora",
        "commit_message": "Upload base QLoRA adapter (Phi-3.5-mini-instruct, GSM8K)",
    },
    {
        "local_path": "/kaggle/input/datasets/naitiksudo/rft-lora/phi35_mini_gsm8k_rft_lora",
        "repo_id": f"{HF_USERNAME}/phi35-mini-gsm8k-rft-lora",
        "commit_message": "Upload RFT-tuned generator adapter",
    },
    {
        "local_path": "/kaggle/input/datasets/naitiksudo/dpo-dataset/phi35_mini_gsm8k_dpo_lora",
        "repo_id": f"{HF_USERNAME}/phi35-mini-gsm8k-dpo-lora",
        "commit_message": "Upload DPO-tuned generator adapter (final generator, 58% best-of-4)",
    },
    {
        "local_path": "/kaggle/input/datasets/naitiksudo/final-results-lama-included/llama32_1b_verifier_lora",
        "repo_id": f"{HF_USERNAME}/llama32-1b-gsm8k-verifier-lora",
        "commit_message": "Upload Llama-3.2-1B binary correctness verifier adapter",
    },
]

DATASET_ITEM = {
    "local_path": "/kaggle/input/datasets/naitiksudo/verifier-training-data/verifier_training_data.jsonl",
    "repo_id": f"{HF_USERNAME}/gsm8k-verifier-training-data",
    "commit_message": "Upload verifier training data (question, candidate, label, source_checkpoint)",
}

# ---------------------------------------------------------------------------
# 3. Push helpers
# ---------------------------------------------------------------------------
def push_adapter(local_path: str, repo_id: str, commit_message: str) -> None:
    path = Path(local_path)
    if not path.exists():
        print(f"[SKIP] {local_path} does not exist — fix the path and re-run this item.")
        return

    print(f"Creating/using model repo: {repo_id}")
    create_repo(repo_id, repo_type="model", exist_ok=True, private=PRIVATE, token=HF_TOKEN)

    print(f"Uploading {local_path} -> {repo_id}")
    api.upload_folder(
        folder_path=str(path),
        repo_id=repo_id,
        repo_type="model",
        commit_message=commit_message,
    )
    print(f"Done: https://huggingface.co/{repo_id}\n")


def push_dataset(local_path: str, repo_id: str, commit_message: str) -> None:
    path = Path(local_path)
    if not path.exists():
        print(f"[SKIP] {local_path} does not exist — fix the path and re-run this item.")
        return

    print(f"Creating/using dataset repo: {repo_id}")
    create_repo(repo_id, repo_type="dataset", exist_ok=True, private=PRIVATE, token=HF_TOKEN)

    print(f"Uploading {local_path} -> {repo_id}")
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=commit_message,
    )
    print(f"Done: https://huggingface.co/datasets/{repo_id}\n")


# ---------------------------------------------------------------------------
# 4. Run
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    for item in MODEL_ITEMS:
        push_adapter(item["local_path"], item["repo_id"], item["commit_message"])

    push_dataset(DATASET_ITEM["local_path"], DATASET_ITEM["repo_id"], DATASET_ITEM["commit_message"])

    print("All uploads attempted — check the links printed above for anything that succeeded,")
    print("and re-run individual items after fixing paths for anything marked [SKIP].")

In [23]:
"""
export_showcase_examples.py

Run this in your Kaggle notebook, in the SAME session where your existing
generate(), verify_arithmetic(), select_best_candidate(), score_candidate(),
and extract_answer() functions are already defined (from your eval cells).
It reuses those exact functions -- nothing reimplemented, nothing guessed.

It runs your real best-of-4 pipeline on a handful of GSM8K test questions and
saves the full output -- every candidate, calculator check results, verifier
score, chosen answer, correctness -- to a JSON file.

WHY THIS EXISTS: Hugging Face now requires a paid plan to create a
live-inference Gradio Space. Instead of live typing, we're building a static
(free, permanent) showcase page that displays genuine, pre-computed pipeline
runs. This script captures the real data for that page.

After running: download showcase_examples.json and send it back so the
static page can be built around actual output instead of placeholders.
"""

import json
from datasets import load_dataset

NUM_EXAMPLES = 8   # how many worked examples to capture
SEED = 123         # different from your eval seed, so these are fresh questions
OUTPUT_PATH = "showcase_examples.json"

dataset = (
    load_dataset("openai/gsm8k", "main", split="test")
    .shuffle(seed=SEED)
    .select(range(NUM_EXAMPLES))
)

showcase = []
for i, ex in enumerate(dataset):
    question, gold_solution = ex["question"], ex["answer"]
    gold = extract_answer(gold_solution)

    candidates = generate(question, n=4)
    checked = [(c, verify_arithmetic(c)) for c in candidates]
    result = select_best_candidate(question, candidates, score_candidate)
    predicted = extract_answer(result["best_candidate"])
    correct = (predicted == gold) if predicted and gold else None

    showcase.append({
        "question": question,
        "gold_answer": gold,
        "candidates": [
            {
                "text": c,
                "calc_valid": r["calc_valid"],
                "num_checks": r["num_checks"],
                "num_errors": r["num_errors"],
            }
            for c, r in checked
        ],
        "best_candidate": result["best_candidate"],
        "best_score": result["best_score"],
        "num_calc_valid": result["num_calc_valid"],
        "num_total": result["num_total"],
        "predicted_answer": predicted,
        "correct": correct,
    })

    print(f"[{i+1}/{NUM_EXAMPLES}] predicted={predicted} gold={gold} correct={correct}")

with open(OUTPUT_PATH, "w") as f:
    json.dump(showcase, f, indent=2)

print(f"\nSaved {len(showcase)} worked examples to {OUTPUT_PATH}")
print("Download this file and send it back so the showcase page can be built from real output.")

[1/8] predicted=621 gold=621 correct=True
[2/8] predicted=5 gold=5 correct=True
[3/8] predicted=7500 gold=7500 correct=True
[4/8] predicted=10 gold=10 correct=True
[5/8] predicted=294 gold=294 correct=True
[6/8] predicted=7 gold=7 correct=True
[7/8] predicted=6 gold=6 correct=True
[8/8] predicted=12 gold=12 correct=True

Saved 8 worked examples to showcase_examples.json
Download this file and send it back so the showcase page can be built from real output.
